In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation
import numpy as np
import sys
import argparse
import os
import glob
import utils_report as ru
import utils_behavior as ub
from utils_behavior import calculate_polarization, calculate_cohesion, theil_index
import cfg
from utils_rnn import plot_cumulative_variance
from cfg import deg2rad, rad2deg

# Check if we're in interactive mode or batch mode
batchmode = False
if "ipykernel_launcher" in sys.argv[0]:
    print("Interactive mode")
else:
    batchmode = True
    print("Batch/CLI mode")
    # Parses the command line arguments below


def get_latest_flat_pkl_file(input_dir="./"):
    pkl_files = glob.glob(input_dir + "/*.pkl")
    pkl_files = [f for f in pkl_files if "flat" in f]
    if not pkl_files:
        raise FileNotFoundError("No .pkl files found in the current directory.")
    latest_pkl_file = max(pkl_files, key=os.path.getctime)
    return latest_pkl_file


# BASE_DIR = "/srv/marl/raaghav/marl_zfish/rmappo-MultiAgentForagingEnv-1_agent/"
# SUB_DIR = "20250808_153214_1_bao_efp_0.05_vd_0.002_fd_10/"  # AI4Sci submission

# BASE_DIR = "/srv/marl/satsingh/zfish/rmappo-MultiAgentForagingEnv-check/"
BASE_DIR = "./results/rmappo-MultiAgentForagingEnv-check/"
SUB_DIR = "20250916_130702_1_collision1_nowalkerbots_seed1/"  # Walkerbot
# SUB_DIR = "20250917_085639_1_2M_GRU_3bots5_food012_align_seed1/"  # 3 bots


outputs_folder = BASE_DIR + SUB_DIR + "outputs/"

if batchmode:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "outputs_folder",
        default=outputs_folder,
        nargs="?",
    )
    args = parser.parse_args()
    outputs_folder = args.outputs_folder

print(f"Using outputs folder: {outputs_folder}")

flat_pkl_file = get_latest_flat_pkl_file(outputs_folder)
print(f"Using .pkl file: {flat_pkl_file}")

dff = pd.read_pickle(flat_pkl_file)
# ru.print_column_shapes(dff)
print("dff.shape", dff.shape)
print("dff.columns", dff.columns)

In [ ]:
dff = dff.sort_values(
    by=["env_id", "episode_index", "agent_id", "time_step"]
).reset_index(drop=True)

print(dff.head())

In [ ]:
dff["position_x"] = dff["position"].apply(lambda pos: pos[0])
dff["position_y"] = dff["position"].apply(lambda pos: pos[1])
dff["num_detected_food"] = dff["detected_food_ids"].apply(lambda x: len(x) if isinstance(x, list) else 0)
dff["num_eaten_food"] = dff["eaten_food_ids"].apply(lambda x: len(x) if isinstance(x, list) else 0)
dff["num_nearby_food"] = dff["nearby_food_ids"].apply(lambda x: len(x) if isinstance(x, list) else 0)
dff["num_binocular_food"] = dff["binocular_food_ids"].apply(lambda x: len(x) if isinstance(x, list) else 0)
dff["has_nearby"] = dff["num_nearby_food"] > 0
dff["orientation_rad"] = dff["orientation"].apply(lambda x: deg2rad * x).apply(lambda x: (x + np.pi) % (2 * np.pi) - np.pi)  # Normalize to [-pi, pi]

viz_columns = [
    "position_x",
    "position_y",
    "orientation",
    "orientation_rad",
    "left_eye_angle",
    "right_eye_angle",
    # 'has_nearby',
    "num_detected_food",
    "num_binocular_food",
    "num_eaten_food",
    "dist_to_wall",
    "move_forward",
    "turn_angle",
    "displacement",
    # 'num_nearby_food',
    "vergence_angle",
    "speed",
    "distance_to_food_hunting",
    "angle_to_food_hunting",
]

for col in viz_columns:
    if col not in dff.columns:
        print(f"Column {col} not found in DataFrame")
        continue
    # print(f"Plotting histogram for column: {col}")
    plt.figure(figsize=(5, 1.5))
    dff[col].hist(bins=50)
    plt.title(col)
    plt.show()

In [ ]:
from utils_behavior import analyze_vergence_during_food_tracking

# Perform the analysis
tracking_sequences_df = analyze_vergence_during_food_tracking(dff)
print(tracking_sequences_df.shape)

tracking_sequences_df.head()

In [ ]:
# Initialize the "hunting" column with False
dff["hunting"] = False

# Iterate through tracking_results to mark hunting time steps
for _, row in tracking_sequences_df.iterrows():
    mask = (
        (dff["env_id"] == row["env_id"]) &
        (dff["episode_index"] == row["episode_index"]) &
        (dff["agent_id"] == row["agent_id"]) &
        (dff["time_step"] >= row["start_time_step"]) &
        (dff["time_step"] < row["start_time_step"] + row["tracking_duration"])
    )
    dff.loc[mask, "hunting"] = True

In [ ]:
dff.columns

In [ ]:
# Cumulative Variance explained by PCs
rnn_states = np.vstack(dff["rnn_states"].values)
pca_full = PCA()
rnn_states_pca = pca_full.fit_transform(rnn_states)
plot_cumulative_variance(
    pca_full,
)

In [ ]:
# Extract RNN states and apply PCA
rnn_states = np.vstack(dff["rnn_states"].values)
pca = PCA(n_components=2)
pca_result = pca.fit_transform(rnn_states)


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

X = np.vstack(dff["rnn_states"].values)
y = dff["hunting"].astype(int).values

lda = LinearDiscriminantAnalysis(n_components=1)  # binary → 1D
X_lda = lda.fit_transform(X, y)

plt.figure(figsize=(6, 4))
plt.hist(X_lda[y==0], bins=50, alpha=0.5, label="Not Hunting")
plt.hist(X_lda[y==1], bins=50, alpha=0.5, label="Hunting")
plt.legend(fontsize=15)
plt.xlabel("LDA Projection", fontsize=15)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

X = np.vstack(dff["rnn_states"].values)
y = dff["hunting"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("AUC:", roc_auc_score(y_test, clf.predict_proba(X_test)[:,1]))

In [ ]:
# Filter successful hunts
successful_hunts = tracking_sequences_df[tracking_sequences_df["outcome"] == "success"]

# Initialize lists to store distances and orientations
distances_to_food = []
orientations_to_food = []
pca_points = []

# Iterate through successful hunts
for _, hunt in successful_hunts.iterrows():
    # Filter the corresponding time steps in dff
    mask = (
        (dff["env_id"] == hunt["env_id"]) &
        (dff["episode_index"] == hunt["episode_index"]) &
        (dff["agent_id"] == hunt["agent_id"]) &
        (dff["time_step"] >= hunt["start_time_step"]) &
        (dff["time_step"] < hunt["start_time_step"] + hunt["tracking_duration"])
    )
    hunt_steps = dff[mask]

    # Get the food position
    food_id = hunt["food_id"]

    # Calculate distance and orientation for each time step
    for t, row in hunt_steps.iterrows():
        idx = t - hunt_steps.index[0]  # Get the relative index for the current time step
        agent_position = row["position"]
        agent_orientation = row["orientation"]
        # agent_orientation = (agent_orientation + np.pi) % (2 * np.pi) - np.pi  # Normalize to [-pi, pi]

        food_positions = hunt_steps["food_positions"].iloc[idx]
        food_ids = hunt_steps["food_ids"].iloc[idx]
        food_position = food_positions[np.where(np.array(food_ids) == food_id)[0][0]]

        # Calculate distance
        distance = np.linalg.norm(np.array(agent_position) - np.array(food_position))
        distances_to_food.append(distance)

        # Calculate orientation
        vector_to_food = np.array(food_position) - np.array(agent_position)
        orientation_to_food = np.arctan2(vector_to_food[1], vector_to_food[0]) - agent_orientation
        orientation_to_food = (orientation_to_food + np.pi) % (2 * np.pi) - np.pi  # Normalize to [-pi, pi]
        orientations_to_food.append(np.rad2deg(orientation_to_food))

        # Store PCA point
        pca_points.append(pca.transform([row["rnn_states"][0]])[0])

# Convert to numpy arrays for plotting
pca_points = np.array(pca_points)
distances_to_food = np.array(distances_to_food)
orientations_to_food = np.array(orientations_to_food)


In [ ]:
# Scatter plot colored by behavior mode
print(dff["hunting"].astype(int).describe())
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(
    pca_result[:, 0], 
    pca_result[:, 1], 
    c=dff["hunting"].astype(int).values,
    # cmap="twilight", 
    alpha=0.7,
    s=3,
)
plt.colorbar(scatter, label="Hunting")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("RNN PCA Colored by Hunting")
plt.show()

In [ ]:
dff_hunting = dff[dff["hunting"]]
print("dff_hunting.shape", dff_hunting.shape)



plt.figure(figsize=(4.5, 3))
pd.Series(distances_to_food).hist(bins=50)
plt.ylabel("Count")
plt.xlabel("Distance to Food (meters)")
plt.title("Distance to Food when Hunting (success)")
plt.show()

# Scatter plot colored by distance to food
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(pca_points[:, 0], pca_points[:, 1], c=distances_to_food, cmap="viridis", alpha=0.7, s=3,)
plt.colorbar(scatter, label="Distance to Food")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Distance to Food when Hunting (success)")
plt.show()

In [ ]:
# pd.Series(orientations_to_food)[:100].plot()
plt.figure(figsize=(4.5, 3))
pd.Series(orientations_to_food).hist(bins=50)
plt.ylabel("Count")
plt.xlabel("Orientation to Food (degrees)")
plt.title("Orientation to Food when Hunting (success)")

orientations_to_food_rad = np.deg2rad(orientations_to_food)

In [ ]:
# Scatter plot colored by orientation to food
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(
    pca_points[:, 0],
    pca_points[:, 1],
    c=orientations_to_food_rad,
    cmap="twilight",
    alpha=0.7,
    vmin=-np.pi,
    vmax=np.pi,
    s=3,
)
plt.colorbar(scatter, label="Orientation to Food (radians)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Orientation to Food when Hunting (success)")
plt.show()

In [ ]:
# Scatter plot colored by agent orientation
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(
    pca_result[:, 0], 
    pca_result[:, 1], 
    # c=np.log1p(dff["orientation"]-min(dff["orientation"])), 
    c=dff["orientation_rad"],
    cmap="twilight", 
    alpha=0.7,
    s=3,
)
plt.colorbar(scatter, label="Agent Orientation (radians)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("RNN PCA Colored by Agent Orientation")
plt.show()



In [ ]:

# Scatter plot colored by agent orientation -- Hunting only 
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(
    pca_result[:, 0][dff["hunting"]], 
    pca_result[:, 1][dff["hunting"]], 
    # c=np.log1p(dff["orientation"]-min(dff["orientation"])), 
    c=dff["orientation_rad"][dff["hunting"]],
    cmap="twilight", 
    alpha=0.7,
    s=3,
)
plt.colorbar(scatter, label="Agent Orientation (radians)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("RNN PCA Colored by Agent Orientation -- Hunting only ")
plt.show()


In [ ]:
plt.figure(figsize=(4.5, 3))
pd.Series(dff["dist_to_wall"]).hist(bins=50)
plt.ylabel("Count")
plt.xlabel("Distance to Wall (meters)")
plt.show()

# Scatter plot colored by distance to food
plt.figure(figsize=(4.5, 3))
scatter = plt.scatter(
    pca_result[:, 0],
    pca_result[:, 1],
    c=dff["dist_to_wall"],
    cmap="viridis",
    alpha=0.5,
    s=2,
)
plt.colorbar(scatter, label="Distance to Wall")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
# plt.title("Distance to Wall")
plt.show()

In [ ]:
# Walkerbot only
if "walkerbot_visible" in dff.columns:
    dff["pc1"] = pca_result[:, 0]
    dff["pc2"] = pca_result[:, 1]
    dff_walkerbots_visible = dff[dff["walkerbot_visible"]]
    print("dff_walkerbots_visible.shape", dff_walkerbots_visible.shape)
    print("dff.shape", dff.shape)


    # Scatter plot colored by 'distance_to_closest_walker', 'angle_to_closest_walker',
    for var in ['distance_to_closest_walker', 'angle_to_closest_walker',]:
        plt.figure(figsize=(4.5, 3))
        scatter = plt.scatter(
            dff_walkerbots_visible["pc1"],
            dff_walkerbots_visible["pc2"],
            c=dff_walkerbots_visible[var],
            cmap="Greens_r" if var == "distance_to_closest_walker" else "twilight",
            alpha=0.7,
            s=3,
        )
        plt.colorbar(scatter, label=var)
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title(f"RNN PCA Colored by {var}")
        plt.show()